In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers , models
import random
from collections import deque
from kelimelik_parametric_env import KelimelikDQNEnv, actions_84
import kelimelik_engine1 as ke
from collections import Counter

In [2]:
with open(r"C:\Users\Cenk Toker\OneDrive\Desktop\python codes\rl_kelimelik\turkce_kelime_listesi.txt", "r", encoding="utf-8") as f:
    sozluk1 = f.read().splitlines()
sozluk = [a.replace("i","İ").replace("ı","I").replace("ğ","Ğ")
            .replace("ş","Ş").replace("ö","Ö").replace("ü","Ü")
            .replace("ç","Ç").upper() for a in sozluk1]

In [3]:
tahta_puanlari2=np.array([
        [0 , 0 , 6 , 0 , 0 , 2 , 0 , 0 , 0 , 2 , 0 , 0 , 6 , 0 , 0 ],
        [0 , 3 , 0 , 0 , 0 , 0 , 2 , 0 , 2 , 0 , 0 , 0 , 0 , 3 , 0 ] ,
        [6 , 0 , 0 , 0 , 0 , 0 , 0 , 4 , 0 , 0 , 0 , 0 , 0 , 0 , 6 ] ,
        [0 , 0 , 0 , 4 , 0 , 0 , 0 , 0 , 0 , 0 , 0 , 4 , 0 , 0 , 0 ],
        [0 , 0 , 0 , 0 , 3 , 0 , 0 , 0 , 0 , 0 , 3 , 0 , 0 , 0 , 0 ],
        [2 , 0 , 0 , 0 , 0 , 2 , 0 , 0 , 0 , 2 , 0 , 0 , 0 , 0 , 2],
        [0 , 2 , 0 , 0 , 0 , 0 , 2 , 0 , 2 , 0 , 0 , 0 , 0 , 2 ,0] ,
        [0 , 0 , 4 , 0 , 0 , 0 , 0 , 0 , 0 , 0 , 0 , 0 , 4 , 0 ,0],
        [0 , 2 , 0 , 0 , 0 , 0 , 2 , 0 , 2 , 0 , 0 , 0 , 0 , 2 ,0] ,
        [2 , 0 , 0 , 0 , 0 , 2 , 0 , 0 , 0 , 2 , 0 , 0 , 0 , 0 , 2],
        [0 , 0 , 0 , 0 , 3 , 0 , 0 , 0 , 0 , 0 , 3 , 0 , 0 , 0 , 0 ],
        [0 , 0 , 0 , 4 , 0 , 0 , 0 , 0 , 0 , 0 , 0 , 4 , 0 , 0 , 0 ],
        [6 , 0 , 0 , 0 , 0 , 0 , 0 , 4 , 0 , 0 , 0 , 0 , 0 , 0 , 6 ],
        [0 , 3 , 0 , 0 , 0 , 0 , 2 , 0 , 2 , 0 , 0 , 0 , 0 , 3 , 0 ] ,
        [0 , 0 , 6 , 0 , 0 , 2 , 0 , 0 , 0 , 2 , 0 , 0 , 6, 0 , 0 ]]
          )

harf_stogu = {
    'A': 12, 'B': 2, 'C': 2, 'Ç': 2, 'D': 2, 'E': 8, 'F': 1, 'G': 1, 'Ğ': 1,
    'H': 1, 'I': 4, 'İ': 7, 'J': 1, 'K': 7, 'L': 7, 'M': 4, 'N': 5, 'O': 3,
    'Ö': 1, 'P': 1, 'R': 6, 'S': 3, 'Ş': 2, 'T': 5, 'U': 3, 'Ü': 2, 'V': 1,
    'Y': 2, 'Z': 2
}

In [4]:
def trim_dictionary(words, stock):
    filtered = []
    for kelime in words:
        cnt = Counter(kelime)
        uygun = True
        for harf, adet in cnt.items():
            # stokta olmayan harfleri (örneğin rakam/tire vb.) doğrudan ele
            if harf not in stock:
                uygun = False
                break
            if adet > stock[harf] + 1:
                uygun = False
                break
        if uygun:
            filtered.append(kelime)
    return filtered

In [5]:
sozluk=trim_dictionary(sozluk, harf_stogu)

In [6]:
def remove_long_words(words, limit=10):
    return [w for w in words if len(w.strip()) <= limit]

In [7]:
sozluk = remove_long_words(sozluk, limit=11)

In [8]:
# -*- coding: utf-8 -*-
"""
Kelimelik DQN Trainer (DEBUG'li)
--------------------------------
Bu dosya, mevcut ortam (kelimelik_parametric_env.KelimelikEnv) ve
model (dqn_model_keras.build_dqn_model) ile DQN eğitimi yapar.
Ayrıca Ajan2'nin seçeneklerinin neden giderek azaldığını teşhis edebilmek için
adım adım kapsamlı DEBUG çıktıları içerir.

NOT:
- Ortamın step() fonksiyonundan dönen info sözlüğünde en azından
  {"kelime", "puan", "agirliklar"} alanları beklenir.
- Eğer ortam ek tanılama bilgileri (ör. opsiyon sayıları) sağlıyorsa
  bu script onları da otomatik olarak loglar.
"""

import os
import random
from collections import deque
import numpy as np
import tensorflow as tf
from tensorflow import keras

# Yerel modüller
from kelimelik_parametric_env import KelimelikDQNEnv, actions_84
from dqn_model_keras import build_dqn_model, encode_obs_dict

# ---- Genel ayarlar ----
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
np.set_printoptions(suppress=True)

# ---- Hiperparametreler ----
GAMMA = 0.99
LR = 1e-3
BATCH_SIZE = 64
MEM_SIZE = 50_000
WARMUP_STEPS = 2_000
TRAIN_STEPS_PER_ENV_STEP = 1
TARGET_UPDATE_EVERY = 1_000

# Epsilon programı (lineer decay)
EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY_STEPS = 30_000

MAX_EPISODES = 1_000
MAX_STEPS_PER_EPISODE = 60

'''
# Tekrarlanabilirlik (isteğe bağlı)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
'''

# ---- Yardımcılar ----
class ReplayBuffer:
    def __init__(self, capacity):
        self.buf = deque(maxlen=capacity)
    def __len__(self):
        return len(self.buf)
    def push(self, s, a, r, s2, done):
        self.buf.append((s, a, r, s2, done))
    def sample(self, batch_size):
        idx = np.random.choice(len(self.buf), size=batch_size, replace=False)
        batch = [self.buf[i] for i in idx]
        s, a, r, s2, d = map(np.array, zip(*batch))
        return s, a, r.astype(np.float32), s2, d.astype(np.float32)


def epsilon_by_step(t):
    if t >= EPS_DECAY_STEPS:
        return EPS_END
    return EPS_START - (EPS_START - EPS_END) * (t / EPS_DECAY_STEPS)


def board_fill_ratio(board_tensor):
    """board: (15,15,30) one-hot. En az bir kanalı 1 olan hücre sayısı / 225."""
    try:
        filled = (board_tensor.sum(axis=-1) > 0).astype(np.float32)
        return float(filled.sum() / (board_tensor.shape[0] * board_tensor.shape[1]))
    except Exception:
        return np.nan


# ---- Model ve hedef ağ ----
q_net = build_dqn_model()
q_net.compile(optimizer=keras.optimizers.Adam(LR), loss="mse")

# Hedef ağ (kopya)
target_net = build_dqn_model()
target_net.set_weights(q_net.get_weights())

# ---- Ortam ----
env = KelimelikDQNEnv(sozluk, tahta_puanlari2, harf_stogu)
rb = ReplayBuffer(MEM_SIZE)

global_step = 0
best_eval = -np.inf

# ---- Eğitim döngüsü ----
for ep in range(1, MAX_EPISODES + 1):
    obs_dict, info = env.reset()
    ep_reward = 0.0
    last_word = None

    print("\n" + "="*90)
    #print(f"[EP {ep}] Başladı | Stok: {sum(env.stok.values())} harf | Raf: {env.elde}")
    print("="*90)

    for t in range(1, MAX_STEPS_PER_EPISODE + 1):
        global_step += 1
        eps = epsilon_by_step(global_step)

        state_vec = encode_obs_dict(obs_dict).reshape(1, -1)

        if np.random.rand() < eps:
            action_id = np.random.randint(len(actions_84))
            act_src = "random"
        else:
            qvals = q_net.predict(state_vec, verbose=0)
            action_id = int(np.argmax(qvals[0]))
            act_src = "policy"
        ####################
        #Burada adım atıyor#
        ####################
        next_obs_dict, reward, done, truncated, step_info = env.step(action_id)
        ep_reward += float(reward)
        

        next_state_vec = encode_obs_dict(next_obs_dict).reshape(1, -1)
        rb.push(state_vec[0], action_id, reward, next_state_vec[0], float(done or truncated))

        fill = board_fill_ratio(next_obs_dict.get("board"))
        #kelime = step_info.get("kelime_biz") if isinstance(step_info, dict) else None
        kelime_list = step_info.get("kelime_biz") if isinstance(step_info, dict) else None
        puan = step_info.get("puan_biz") if isinstance(step_info, dict) else None
        w = step_info.get("agirliklar") if isinstance(step_info, dict) else None
        if isinstance(w, tuple):
            w = tuple(round(x, 2) for x in w)

        

        #if isinstance(kelime, list):
         #   ops_sayisi = len(kelime)
        #elif kelime:
         #   ops_sayisi = 1
        #else:
         #   ops_sayisi = 0
        best_move = None
        if isinstance(kelime_list, list) and len(kelime_list) > 0:
            best_move = kelime_list[0]          # örn: ('İT', 8, 12, 'v', ...)
            ops_sayisi = len(kelime_list)
        elif kelime_list:
            best_move = kelime_list
            ops_sayisi = 1
        else:
            ops_sayisi = 0
        
        # best_move tuple ise kelime adı best_move[0]
        best_move_str = "PASS" if best_move is None else best_move

        stok_kalan = sum(env.stok.values())
        print(
            f"[EP {ep:03d} | T {t:02d}] eps={eps:.3f} act={action_id:02d} ({act_src}) "
            f"| opsiyon_sayısı={ops_sayisi} | kelime={best_move_str} puan={puan} | stok={stok_kalan} | raf={env.elde}"
        )
        print(f"           doluluk={fill:.3f} | ağırlıklar={w}")
        print(f"Biz: {env.own_score} | Rakip: {env.opp_score} | Turn: {env.turn}")
        print(f"##############################################################################")

        if len(rb) >= WARMUP_STEPS:
            for _ in range(TRAIN_STEPS_PER_ENV_STEP):
                s, a, r, s2, d = rb.sample(BATCH_SIZE)
                target_q = q_net.predict(s, verbose=0)
                next_q   = target_net.predict(s2, verbose=0)
                max_next = np.max(next_q, axis=1)
                y = r + (1.0 - d) * GAMMA * max_next
                for i, act in enumerate(a):
                    target_q[i, act] = y[i]
                q_net.train_on_batch(s, target_q)

        if global_step % TARGET_UPDATE_EVERY == 0:
            target_net.set_weights(q_net.get_weights())
            print(f"[SYNC] target_net güncellendi @ step {global_step}")
        #env.render()
        obs_dict = next_obs_dict
        last_word = best_move

        if done or truncated:
            break

    
    print(f"[EP {ep}] Bitti | Adım: {t} | Toplam ödül: {ep_reward:.1f} | Son kelime: {last_word}")

    if ep_reward > best_eval:
        best_eval = ep_reward
        try:
            q_net.save("kelimelik_dqn_debug_best.h5")
            print(f"[SAVE] Yeni en iyi ödül {best_eval:.1f} — model kaydedildi.")
        except Exception as e:
            print("[WARN] Model kaydedilemedi:", e)

print("\nEğitim tamamlandı.")


---------------------------------------------
[EP 001 | T 01] eps=1.000 act=68 (random) | opsiyon_sayısı=104 | kelime=('YIĞMA', 11, 7, 'v', 32, 6.4, 160.97, 5.03) puan=32 | stok=76 | raf=['Ü', 'H', 'T', 'D', 'İ', 'E', 'S']
           doluluk=0.058 | ağırlıklar=(0.5, 0.17, 0.0, 0.33)
Biz: 32 | Rakip: 19 | Turn: 1
##############################################################################
---------------------------------------------
[EP 001 | T 02] eps=1.000 act=81 (random) | opsiyon_sayısı=170 | kelime=('ALTAYİST', 7, 7, 'h', 22, 2.75, 79.44, 3.61) puan=22 | stok=68 | raf=['Ü', 'H', 'D', 'E', 'A', 'K', 'L']
           doluluk=0.093 | ağırlıklar=(0.83, 0.0, 0.17, 0.0)
Biz: 54 | Rakip: 52 | Turn: 2
##############################################################################
---------------------------------------------
[EP 001 | T 03] eps=1.000 act=71 (random) | opsiyon_sayısı=165 | kelime=('HÜDA', 10, 4, 'v', 22, 5.5, 89.3, 4.06) puan=22 | stok=62 | raf=['E', 'A', 'K', 'L', 'F', '

KeyboardInterrupt: 